In [1]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import kagglehub

In [2]:
path = kagglehub.dataset_download("lzyacht/proteinsubcellularlocalization")

print("Path to dataset files:", path)



Using Colab cache for faster access to the 'proteinsubcellularlocalization' dataset.
Path to dataset files: /kaggle/input/proteinsubcellularlocalization


In [3]:
cp -r $path 'sample_data'

In [ ]:
# 1. Define how to read a single row
class ProteinDataset(Dataset):
    def __init__(self, csv_file):
        self.df = pd.read_csv(csv_file)
        
        # Build Vocab
        self.aa_to_int = {aa: i+1 for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}
        print(f"Vocab size: {len(self.aa_to_int)}")
        self.aa_to_int['<PAD>'] = 0 
        
        # Build Labels
        self.label_to_int = {label: i for i, label in enumerate(self.df['label'].unique())}
        
    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        # Convert string to numbers
        seq = [self.aa_to_int.get(c, 0) for c in self.df.iloc[idx]['sequence']]
        label = self.label_to_int[self.df.iloc[idx]['label']]
        return torch.tensor(seq), torch.tensor(label)

In [32]:
# 2. Define how to stack them into a batch
def pad_collate(batch):
    seqs = pad_sequence([item[0] for item in batch], batch_first=True, padding_value=0)
    labels = torch.stack([item[1] for item in batch])
    return seqs, labels

In [33]:
#3. Load the datasets
dataset = ProteinDataset('sample_data/proteinsubcellularlocalization/proteins.csv')
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=pad_collate)

In [34]:
batch_seqs, batch_labels = next(iter(loader))
print(f"Batch Sequences Shape: {batch_seqs.shape} -> (batch_size, padded_length)")
print(f"Batch Labels Shape: {batch_labels.shape}")
print(f"Sample Padded Tensor:\n {batch_seqs[0][:20]}... (first 20 tokens)")

Batch Sequences Shape: torch.Size([4, 2195]) -> (batch_size, padded_length)
Batch Labels Shape: torch.Size([4])
Sample Padded Tensor:
 tensor([11, 16, 12, 18, 16,  1,  6, 12, 20, 11, 18,  7, 16, 18,  6, 16, 20, 18,
         1, 18])... (first 20 tokens)
